# Brouillon Julien — partie 2A (données) et 2B (modèles)

Le **test n'est pas ouvert** ici : tout se compare sur la validation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
notifs = pd.read_csv('rev-notifications.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])

# Règles fixées en partie 1
WINDOW = pd.Timedelta(days=60)
MIN_TENURE = pd.Timedelta(days=60)
CUTOFFS = {'fit': pd.Timestamp('2018-11-14'),
           'validation': pd.Timestamp('2019-01-14'),
           'test': pd.Timestamp('2019-03-15')}
X_OFFRE, BUDGET = 5, 500          # € par offre, offres par mois (hypothèses)
INTERCHANGE, USD_TO_EUR = 0.003, 0.89

## 1. Audit des données

| Constat | Décision |
|---|---|
| 202 transactions > 100 000 $, jusqu'à 85 milliards $ (virements) | erreurs de saisie : montants **plafonnés** au 99e percentile, calculé sur la période avant la 1re coupure |
| 58 604 transactions à 0 $ | paiements annulés, en attente ou refusés (REVERTED, PENDING, DECLINED) : gardés comme **signal d'échec**, sans effet sur les montants |
| Infos marchand vides à 42 % | vides seulement hors paiements carte (complètes à 100 % pour les paiements carte) : utilisées sur les paiements carte |
| Opt-in marketing inconnus pour 34 % des clients | inconnu ≠ non : on ajoute une colonne « inconnu » |
| `num_referrals`, `num_successful_referrals` toujours à 0 | retirées |
| 664 clients sans aucune transaction | jamais actifs, donc jamais dans le tableau |
| 715 notifications datées avant l'inscription | une par client, moins de 19 h avant, notifications d'onboarding : décalage d'horloge, gardées |
| `plan` | **fuite** (valeur au jour de l'export) : retirée |

In [2]:
# Vérifications de l'audit
print('Transactions > 100 000 $ :', (tx.amount_usd > 1e5).sum())
print('Transactions à 0 $ :', (tx.amount_usd == 0).sum())
print(tx[tx.amount_usd == 0].transactions_state.value_counts().to_dict())
print('Infos marchand vides (paiements carte) :',
      f"{tx[tx.transactions_type == 'CARD_PAYMENT'].ea_merchant_mcc.isna().mean():.1%}")
print('Opt-in inconnus :', f"{users.attributes_notifications_marketing_push.isna().mean():.1%}")
print('Colonnes constantes :', [c for c in users.columns if users[c].nunique() <= 1])
print('Clients sans transaction :', (~users.user_id.isin(tx.user_id)).sum())

# Plafond des montants : 99e percentile, calculé uniquement sur le passé de la 1re coupure
CAP = tx.loc[(tx.created_date < CUTOFFS['fit']) & (tx.amount_usd > 0), 'amount_usd'].quantile(0.99)
tx['montant'] = tx.amount_usd.clip(upper=CAP)
print(f'Plafond des montants : {CAP:.0f} $')

Transactions > 100 000 $ : 202
Transactions à 0 $ : 58604
{'REVERTED': 31673, 'PENDING': 10747, 'DECLINED': 8180, 'COMPLETED': 8002, 'CANCELLED': 2}


Infos marchand vides (paiements carte) : 0.0%
Opt-in inconnus : 34.0%
Colonnes constantes : ['num_referrals', 'num_successful_referrals']
Clients sans transaction : 664
Plafond des montants : 851 $


## 2. `build_dataset(cutoff)` : une ligne par client actif

Toutes les features utilisent **uniquement** ce qui s'est passé avant la coupure ; le label uniquement ce qui se passe après.

In [3]:
ECHEC = ['DECLINED', 'FAILED', 'REVERTED', 'CANCELLED']
TOP_PAYS = users.country.value_counts().index[:5]

def build_dataset(cutoff):
    passe = tx[tx.created_date < cutoff]
    futur = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + WINDOW)]
    p90 = passe[passe.created_date >= cutoff - pd.Timedelta(days=90)]
    p30 = passe[passe.created_date >= cutoff - pd.Timedelta(days=30)]
    ok90 = p90[p90.transactions_state == 'COMPLETED']
    carte90 = ok90[ok90.transactions_type == 'CARD_PAYMENT']

    # clients actifs (une transaction dans les 60 jours avant) et ancienneté >= 60 jours
    actifs = passe[passe.created_date >= cutoff - WINDOW].user_id.unique()
    u = users.set_index('user_id').loc[actifs]
    d = pd.DataFrame(index=u.index)
    d['anciennete_j'] = (cutoff - u.created_date).dt.days
    d = d[d.anciennete_j >= MIN_TENURE.days]
    idx = d.index
    par_client = lambda s: s.reindex(idx).fillna(0)

    # Activité
    d['recence_j'] = (cutoff - passe.groupby('user_id').created_date.max().reindex(idx)).dt.days
    d['nb_tx_30j'] = par_client(p30.groupby('user_id').size())
    d['nb_tx_90j'] = par_client(p90.groupby('user_id').size())
    d['tendance'] = d.nb_tx_30j / (d.nb_tx_90j / 3 + 1)

    # Montants (transactions terminées, plafonnées)
    d['montant_90j'] = par_client(ok90.groupby('user_id').montant.sum())
    d['panier_moyen_90j'] = par_client(ok90.groupby('user_id').montant.mean())

    # Types de transactions (part sur 90 jours)
    for t in ['CARD_PAYMENT', 'TRANSFER', 'TOPUP', 'EXCHANGE', 'ATM']:
        d[f'part_{t.lower()}'] = par_client(p90[p90.transactions_type == t].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Échecs
    d['taux_echec_90j'] = par_client(p90[p90.transactions_state.isin(ECHEC)].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Marchands (paiements carte uniquement)
    d['nb_pays_marchands_90j'] = par_client(carte90.groupby('user_id').ea_merchant_country.nunique())
    d['nb_categories_90j'] = par_client(carte90.groupby('user_id').ea_merchant_mcc.nunique())

    # Notifications reçues sur 90 jours
    n90 = notifs[(notifs.created_date < cutoff) & (notifs.created_date >= cutoff - pd.Timedelta(days=90))]
    d['nb_notifs_90j'] = par_client(n90.groupby('user_id').size())
    d['taux_echec_notifs_90j'] = par_client(n90[n90.status == 'FAILED'].groupby('user_id').size()) / d.nb_notifs_90j.clip(lower=1)

    # Profil
    d['age'] = cutoff.year - u.birth_year.reindex(idx)
    d['crypto'] = u.user_settings_crypto_unlocked.reindex(idx)
    d['nb_contacts'] = u.num_contacts.reindex(idx)
    for canal in ['push', 'email']:
        col = u[f'attributes_notifications_marketing_{canal}'].reindex(idx)
        d[f'optin_{canal}'] = col.fillna(0)
        d[f'optin_{canal}_inconnu'] = col.isna().astype(int)
    pays = u.country.reindex(idx).where(u.country.reindex(idx).isin(TOP_PAYS), 'autre')
    d = d.join(pd.get_dummies(pays, prefix='pays', dtype=int))

    # Valeur du client (€ par an) : frais + commission carte, sur 90 jours, annualisée
    frais = par_client(ok90[ok90.transactions_type == 'FEE'].groupby('user_id').montant.sum())
    carte = par_client(carte90.groupby('user_id').montant.sum())
    d['valeur_an_eur'] = (frais + INTERCHANGE * carte) * 4 * USD_TO_EUR

    # Label
    d['churn'] = (~d.index.isin(futur.user_id)).astype(int)
    return d.reset_index()

fit = build_dataset(CUTOFFS['fit'])
val = build_dataset(CUTOFFS['validation'])
print(f'fit : {len(fit)} clients, churn {fit.churn.mean():.1%}, {fit.shape[1] - 2} colonnes')
print(f'validation : {len(val)} clients, churn {val.churn.mean():.1%}')

fit : 8241 clients, churn 16.2%, 31 colonnes
validation : 10512 clients, churn 14.1%


## 3. Tableau anti-fuite : « le jour de la coupure, est-ce que je connais cette valeur ? »

| Colonne(s) | Connue à la coupure ? | Décision |
|---|---|---|
| activité, montants, types, échecs, marchands, notifications | oui : calculées uniquement sur les transactions avant la coupure | gardées |
| ancienneté, âge, pays | oui : ne changent pas (ou peu) | gardées |
| `crypto`, `nb_contacts`, opt-in | **partiellement** : photo prise à l'export (mai 2019), pas à la coupure | gardées, **limite à signaler** |
| `plan` | **non** : plan au jour de l'export, peut avoir changé à cause du départ | **retirée** |
| `num_referrals`, `num_successful_referrals` | toujours 0 | retirées |
| `valeur_an_eur` | oui : calculée sur les 90 jours avant | utilisée pour la décision, pas comme feature du modèle |

## 4. Qu'est-ce qui va avec le churn ?

Pour chaque feature, sur les clients du **fit** : taux de churn chez les 25 % les plus bas et les 25 % les plus hauts (0 contre 1 pour les colonnes oui/non).

⚠️ Ce tableau montre ce qui **va avec** le churn, pas ce qui le **cause**.

In [4]:
FEATURES = [c for c in fit.columns if c not in ['user_id', 'churn', 'valeur_an_eur']]

rows = []
for f in FEATURES:
    x = fit[f]
    if x.nunique() <= 2:
        bas, haut = fit[x == x.min()].churn.mean(), fit[x == x.max()].churn.mean()
        lb, lh = f'= {x.min():g}', f'= {x.max():g}'
    else:
        q1, q3 = x.quantile([0.25, 0.75])
        if q1 == q3:   # trop de valeurs identiques : 0 contre > 0
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x > q1].churn.mean(), f'≤ {q1:g}', f'> {q1:g}'
        else:
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x >= q3].churn.mean(), f'≤ {q1:.3g}', f'≥ {q3:.3g}'
    rows.append({'feature': f, 'bas': lb, 'churn si bas': f'{bas:.0%}',
                 'haut': lh, 'churn si haut': f'{haut:.0%}', 'écart (points)': round(100 * (haut - bas))})

lecture = pd.DataFrame(rows).sort_values('écart (points)', key=abs, ascending=False)
print(f'Taux de churn moyen (fit) : {fit.churn.mean():.0%}')
lecture

Taux de churn moyen (fit) : 16%


,feature,bas,churn si bas,haut,churn si haut,écart (points)
1,recence_j,≤ 1,2%,≥ 19,41%,38
2,nb_tx_30j,≤ 1,39%,≥ 25,2%,-37
3,nb_tx_90j,≤ 11,34%,≥ 73,4%,-30
4,tendance,≤ 0.283,36%,≥ 1.28,9%,-28
5,montant_90j,≤ 216,31%,≥ 2.77e+03,6%,-25
14,nb_categories_90j,≤ 2,30%,≥ 12,6%,-24
13,nb_pays_marchands_90j,≤ 1,26%,≥ 3,6%,-20
19,nb_contacts,≤ 0,27%,≥ 21,8%,-19
28,pays_RO,= 0,17%,= 1,7%,-10
26,pays_IE,= 0,17%,= 1,8%,-9


### Lecture (brouillon, à reformuler avec nos mots)

Taux de churn moyen (fit, novembre 2018) : 16 %.

1. **Le désengagement est le signal n°1.** Rien depuis 19 jours ou plus : 41 % de churn, contre 2 % pour une utilisation la veille. Même chose pour peu de transactions sur 30 jours (39 %) et pour une activité qui ralentit (36 %).
2. **Un usage étroit, c'est un usage fragile.** 2 catégories de commerce ou moins : 30 % de churn, contre 6 % au-delà de 12. Un seul pays marchand : 26 %, contre 6 % au-delà de 3. Revolut utilisé comme compte principal retient mieux qu'un usage d'appoint.
3. **Peu de liens sociaux, plus de départs.** 0 contact : 27 %, contre 8 % au-delà de 21.
4. **Le pays compte un peu.** Royaume-Uni : 21 % ; Roumanie et Irlande : 7 à 8 %.
5. **Ce qui compte peu :** taux d'échec des paiements (+4 points), âge, notifications, opt-in.

⚠️ Ce sont des associations, pas des causes. Un client qui part arrête d'utiliser sa carte, donc il a peu de catégories : c'est un **symptôme**, pas forcément une raison.

## 5. Modèles : 5 features simples contre toutes les features (validation)

In [5]:
SIMPLES = ['anciennete_j', 'recence_j', 'nb_tx_30j', 'nb_tx_90j', 'tendance']

def entraine(cols):
    logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0)
    return {f'logistique ({len(cols)} f.)': logit.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1],
            f'arbres ({len(cols)} f.)': trees.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1]}

scores = {'baseline (récence)': val.recence_j}
scores.update(entraine(SIMPLES))
scores.update(entraine(FEATURES))

## 6. Décision en euros : classer par risque, ou par perte attendue ?

- **Classement par risque** : on cible les 500 clients avec le `p` le plus haut.
- **Classement par perte attendue** : on cible les 500 clients avec le `p × valeur_an_eur` le plus haut (les gros clients risqués d'abord).

Gain = Σ (churners ciblés × Z × leur valeur) − 500 × X. **Z est inconnu** : on montre 30 % et 50 %.

In [6]:
def gain(cibles, z):
    return (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * X_OFFRE

rows = []
for nom, s in scores.items():
    v = val.assign(p=s)
    for critere, cle in [('risque', v.p), ('perte attendue', v.p * v.valeur_an_eur)]:
        if nom.startswith('baseline') and critere == 'perte attendue':
            continue
        cibles = v.loc[cle.nlargest(BUDGET).index]
        rows.append({'méthode': nom, 'classement': critere,
                     'PR-AUC': round(average_precision_score(val.churn, s), 3),
                     'churners ciblés': int(cibles.churn.sum()),
                     'gain € Z=30%': round(gain(cibles, 0.30)),
                     'gain € Z=50%': round(gain(cibles, 0.50))})
pd.DataFrame(rows)

,méthode,classement,PR-AUC,churners ciblés,gain € Z=30%,gain € Z=50%
0,baseline (récence),risque,0.385,229,-2273,-2122
1,logistique (5 f.),risque,0.405,239,-2217,-2028
2,logistique (5 f.),perte attendue,0.405,92,-1191,-318
3,arbres (5 f.),risque,0.427,275,-2097,-1828
4,arbres (5 f.),perte attendue,0.427,97,-1171,-284
5,logistique (30 f.),risque,0.450,275,-2234,-2056
6,logistique (30 f.),perte attendue,0.450,98,-1200,-333
7,arbres (30 f.),risque,0.465,291,-2078,-1796
8,arbres (30 f.),perte attendue,0.465,100,-1127,-211


### Lecture (brouillon)

1. **Plus de features, meilleur ciblage.** Avec 30 features, les arbres passent de 275 à 291 churners dans le top 500 (PR-AUC 0,427 → 0,465). Ils battent la baseline (229) et la logistique (275).
2. **Ceux qui partent sont ceux qui rapportent le moins.** Classés par risque seul, les 500 clients ciblés valent si peu que l'offre à 5 € perd environ 2 000 € par mois, même si elle retient la moitié des clients.
3. **Classer par perte attendue (`p × valeur`) aide, mais ne suffit pas avec 500 offres** : on perd encore environ 200 € à Z = 50 %.
4. **Prochaine étape :** ne plus fixer 500 offres, envoyer seulement quand `p × Z × valeur > X`.

## 7. Calibration : quand le modèle dit 40 %, est-ce que 40 % partent ?

La règle de rentabilité utilise `p` comme une vraie probabilité : il faut le vérifier (séance 4).

In [7]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(fit[FEATURES], fit.churn)
trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0).fit(fit[FEATURES], fit.churn)
probas = {'logistique': logit.predict_proba(val[FEATURES])[:, 1],
          'arbres': trees.predict_proba(val[FEATURES])[:, 1]}

tranches = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1]
calib = []
for nom, p in probas.items():
    t = pd.DataFrame({'p': p, 'churn': val.churn})
    t['tranche'] = pd.cut(t.p, tranches)
    g = t.groupby('tranche', observed=True).agg(clients=('churn', 'size'), p_moyen=('p', 'mean'), churn_reel=('churn', 'mean'))
    g.insert(0, 'modèle', nom)
    calib.append(g)
calib = pd.concat(calib).round(3)
calib

,modèle,clients,p_moyen,churn_reel
tranche,,,,
"(0.0, 0.05]",logistique,3702,0.018,0.012
"(0.05, 0.1]",logistique,1926,0.073,0.047
"(0.1, 0.2]",logistique,2048,0.143,0.147
"(0.2, 0.3]",logistique,997,0.245,0.250
"(0.3, 0.5]",logistique,1138,0.388,0.375
"(0.5, 0.7]",logistique,582,0.588,0.509
"(0.7, 1.0]",logistique,119,0.747,0.664
"(0.0, 0.05]",arbres,4420,0.018,0.011
"(0.05, 0.1]",arbres,1313,0.074,0.059


**Lecture :** les deux modèles sont bien calibrés jusqu'à 50 % (écarts de 1 à 4 points). Au-dessus, ils se surestiment un peu : la logistique annonce 59 % quand 51 % partent vraiment, les arbres 58 % pour 53 %. Les arbres sont un peu plus fiables. → On peut utiliser `p` dans la règle, en gardant en tête qu'il est légèrement optimiste pour les clients les plus risqués.

## 8. Arbres ou logistique : avantage réel ou chance ?

On répète la comparaison sur 3 périodes (entraînement sur une coupure, évaluation sur la suivante). **Le test du 15/03 n'est jamais utilisé.**

In [8]:
# 60 jours d'écart minimum : le label d'entraînement est connu avant la coupure d'évaluation
periodes = [('2018-07-16', '2018-09-14'), ('2018-09-14', '2018-11-14'), ('2018-11-14', '2019-01-14')]
datasets = {}
rows = []
for c_train, c_eval in periodes:
    for c in [c_train, c_eval]:
        if c not in datasets:
            datasets[c] = build_dataset(pd.Timestamp(c))
    tr, ev = datasets[c_train], datasets[c_eval]
    for nom, modele in [('logistique', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
                        ('arbres', HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0))]:
        p = modele.fit(tr[FEATURES], tr.churn).predict_proba(ev[FEATURES])[:, 1]
        cibles = ev.assign(p=p).nlargest(BUDGET, 'p')
        rows.append({'évaluation': c_eval, 'modèle': nom,
                     'PR-AUC': round(average_precision_score(ev.churn, p), 3),
                     'churners top 500': int(cibles.churn.sum())})
comparaison = pd.DataFrame(rows).pivot(index='évaluation', columns='modèle', values=['PR-AUC', 'churners top 500'])
comparaison

PR-AUC            churners top 500           
modèle     arbres logistique           arbres logistique
évaluation                                              
2018-09-14  0.486      0.445            277.0      269.0
2018-11-14  0.508      0.470            317.0      288.0
2019-01-14  0.465      0.450            291.0      275.0

**Lecture :** les arbres battent la logistique **sur les 3 périodes** : de +8 à +29 churners dans le top 500, et de +0,02 à +0,04 de PR-AUC. L'écart va toujours dans le même sens, donc ce n'est pas de la chance. → **Modèle retenu : arbres boostés.**

Les coupures sont espacées de 60 jours, donc aucun label d'entraînement ne chevauche la période évaluée. La performance varie d'une période à l'autre (277 à 317 churners) : le comportement des clients évolue, il faudra réentraîner le modèle régulièrement.

## 9. Règle de rentabilité : n'envoyer que si `p × Z × valeur > X`

Plafond : 500 offres par mois. Pour chaque prix d'offre `X` et efficacité `Z`, on compte les offres envoyées et le gain sur la validation.

In [9]:
p_val = probas['arbres']   # modèle retenu : mieux calibré et meilleur sur les 3 périodes

rows = []
for x in [1, 2, 3, 5]:
    for z in [0.2, 0.3, 0.5]:
        v = val.assign(p=p_val, gain_attendu=p_val * z * val.valeur_an_eur - x)
        cibles = v[v.gain_attendu > 0].nlargest(BUDGET, 'gain_attendu')
        gain_reel = (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * x
        rows.append({'X (€)': x, 'Z': f'{z:.0%}', 'offres envoyées': len(cibles),
                     'churners ciblés': int(cibles.churn.sum()), 'gain réel (€)': round(gain_reel)})
carte = pd.DataFrame(rows)
carte.pivot(index='X (€)', columns='Z', values=['offres envoyées', 'gain réel (€)'])

offres envoyées           gain réel (€)           
Z                 20%  30%  50%           20%  30%   50%
X (€)                                                   
1                 500  500  500           416  873  1789
2                 313  490  500           143  393  1289
3                 174  313  500            61  214   789
5                  76  150  313            53  114   357

**Lecture :** avec la règle de rentabilité, le gain est **positif dans tous les scénarios**, mais modeste pour une offre chère : à 5 € et Z = 30 %, +114 €/mois sur 150 offres.

- La clé, c'est d'envoyer **moins d'offres, mieux ciblées** : à 5 € et Z = 30 %, 150 offres rapportent +114 €, alors que 500 offres aux plus risqués perdent environ 2 000 €.
- **Le prix de l'offre compte plus que le modèle** : à 2 € (Z = 30 %), +393 € ; à 1 €, +873 €.

**Recommandation provisoire :** on lance sous condition, avec une offre à 2 € maximum. Les arbres ciblent les clients à forte valeur et à risque, et on n'envoie que si c'est rentable client par client. Le premier mois, on teste avec un groupe témoin pour mesurer Z.